In [45]:
import pandas as pd
import numpy as np

In [46]:
df = pd.read_csv("01 covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [47]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [48]:
x = df.iloc[:,0:5]
y = df.drop(x,axis='columns')
print(x.head())
print()
print()
print()
print(y.head())

   age  gender  fever cough     city
0   60    Male  103.0  Mild  Kolkata
1   27    Male  100.0  Mild    Delhi
2   42    Male  101.0  Mild    Delhi
3   31  Female   98.0  Mild  Kolkata
4   65  Female  101.0  Mild   Mumbai



  has_covid
0        No
1       Yes
2        No
3        No
4        No


In [49]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)
print(x_train.head())
print()
print()
print()
print(y_train.head())

    age  gender  fever   cough       city
94   79    Male    NaN  Strong    Kolkata
46   19  Female  101.0    Mild     Mumbai
80   14  Female   99.0    Mild     Mumbai
3    31  Female   98.0    Mild    Kolkata
89   46    Male  103.0  Strong  Bangalore



   has_covid
94       Yes
46        No
80       Yes
3         No
89        No


In [50]:
x_train_2 = x_train.copy()
x_test_2 = x_train.copy()

In [51]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

In [52]:
le = LabelEncoder()
si = SimpleImputer()

In [53]:
y_train.has_covid = le.fit_transform(y_train.has_covid)
y_test.has_covid = le.transform(y_test['has_covid'])

In [54]:
x_train.fever = si.fit_transform(x_train[['fever']])

In [55]:
from sklearn.preprocessing import OrdinalEncoder

In [56]:
oe = OrdinalEncoder()
x_train['cough'] = oe.fit_transform(df.loc[x_train.index, ['cough']])
x_test['cough'] = oe.transform(df.loc[x_test.index, ['cough']])

In [57]:
from sklearn.preprocessing import OneHotEncoder

In [58]:
ohe = OneHotEncoder(drop="first",sparse_output=False).set_output(transform = 'pandas')

In [59]:
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])
x_test_gender_city = ohe.transform(x_test[['gender','city']])

print(x_train_gender_city.shape)
print(x_test_gender_city.shape)

(80, 4)
(20, 4)


In [60]:
x_train.drop(['gender','city'],axis='columns',inplace=True)
x_test.drop(['gender','city'],axis='columns',inplace=True)

In [61]:
x_train = pd.concat([x_train, x_train_gender_city], axis='columns')
x_test = pd.concat([x_test, x_test_gender_city], axis='columns')

In [62]:
x_train

,age,fever,cough,gender_Male,city_Delhi,city_Kolkata,city_Mumbai
94,79,100.887324,1.0,1.0,0.0,1.0,0.0
46,19,101.000000,0.0,0.0,0.0,0.0,1.0
80,14,99.000000,0.0,0.0,0.0,0.0,1.0
3,31,98.000000,0.0,0.0,0.0,1.0,0.0
89,46,103.000000,1.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
34,74,102.000000,0.0,1.0,0.0,0.0,1.0
30,15,101.000000,0.0,1.0,1.0,0.0,0.0
2,42,101.000000,0.0,1.0,1.0,0.0,0.0
17,40,98.000000,1.0,0.0,1.0,0.0,0.0


In [63]:
from sklearn.compose import ColumnTransformer

In [64]:
transformer = [('trans1', SimpleImputer(), ['fever']),
               ('trans2', OrdinalEncoder(categories = [['Mild', 'Strong']]), ['cough']),
               ('trans3', OneHotEncoder(sparse_output = False, drop = 'first'), ['gender', 'city'])
               ]

In [65]:
ct = ColumnTransformer(transformers = transformer, remainder = 'passthrough').set_output(transform = 'pandas')

In [67]:
x_train_trans = ct.fit_transform(x_train_2)

In [68]:
x_train_trans

,trans1__fever,trans2__cough,trans3__gender_Male,trans3__city_Delhi,trans3__city_Kolkata,trans3__city_Mumbai,remainder__age
94,100.887324,1.0,1.0,0.0,1.0,0.0,79
46,101.000000,0.0,0.0,0.0,0.0,1.0,19
80,99.000000,0.0,0.0,0.0,0.0,1.0,14
3,98.000000,0.0,0.0,0.0,1.0,0.0,31
89,103.000000,1.0,1.0,0.0,0.0,0.0,46
...,...,...,...,...,...,...,...
34,102.000000,0.0,1.0,0.0,0.0,1.0,74
30,101.000000,0.0,1.0,1.0,0.0,0.0,15
2,101.000000,0.0,1.0,1.0,0.0,0.0,42
17,98.000000,1.0,0.0,1.0,0.0,0.0,40
